# LOB Pretraining Diagnostic

Loads the rollouts saved by `train_lob.py` during training and plots imagined midprice / spread / imbalance trajectories versus the validation ground truth. Use this to visually confirm the world model is learning LOB dynamics.

**Inputs expected:**
- `/content/drive/MyDrive/drama_lob_checkpoints/` (or local equivalent) containing `rollout_step_*.npy` files and `normalization.json`.
- A validation feature sequence, rebuilt on the fly from `data/validation/` via `lob.backtester.build_timeline`.

Set `CKPT_DIR`, `DATA_VAL_DIR`, and `MARKET_SLUG` in the first cell.

In [ ]:
# Paths + market
import sys, os
from pathlib import Path

# Adjust for your environment (Colab vs local)
SRC_DIR = '/content/Drama/src'        # on Colab
# SRC_DIR = r'C:\Users\ruuud\algoverse\Drama\src'  # local
CKPT_DIR = '/content/drive/MyDrive/drama_lob_checkpoints'
DATA_VAL_DIR = '/content/Drama/data/validation'
MARKET_SLUG = None   # None -> pick the longest-lived in the validation split

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print('SRC_DIR     :', SRC_DIR)
print('CKPT_DIR    :', CKPT_DIR)
print('DATA_VAL_DIR:', DATA_VAL_DIR)

In [ ]:
# Build validation features from raw LOB snapshots
import logging
logging.basicConfig(level=logging.WARNING)

from lob.backtester import build_timeline
from envs.lob_features import (
    extract_features, pick_longest_market,
    load_normalization, apply_normalization,
    K_LEVELS, F_LEVEL, F_TICK, FEATURE_DIM_FLAT,
)

bt = build_timeline(data_dir=Path(DATA_VAL_DIR), hours=1.0)
slug = MARKET_SLUG or pick_longest_market(bt)
print('market:', slug)

seq = extract_features(bt.timeline, slug)
stats = load_normalization(Path(CKPT_DIR) / 'normalization.json')
seq_n = apply_normalization(seq, stats)
print('validation ticks:', seq.midprice.shape[0])

In [ ]:
# Plot raw validation midprice / spread / imbalance
import matplotlib.pyplot as plt
import numpy as np

LEVEL_FLAT = K_LEVELS * F_LEVEL
mid = seq.per_tick[:, 0]      # raw mid (not normalised)
spread = seq.per_tick[:, 1]
imbalance = seq.per_tick[:, 3]

fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
axes[0].plot(mid); axes[0].set_ylabel('mid (YES)')
axes[1].plot(spread); axes[1].set_ylabel('spread')
axes[2].plot(imbalance); axes[2].set_ylabel('top imbalance')
axes[2].set_xlabel('tick (1s)')
fig.suptitle(f'Validation truth: {slug}')
plt.show()

In [ ]:
# Load the latest saved imagined rollout(s) and plot predicted trajectories
import glob

roll_paths = sorted(glob.glob(str(Path(CKPT_DIR) / 'rollout_step_*.npy')),
                    key=lambda p: int(Path(p).stem.split('_')[-1]))
print(f'found {len(roll_paths)} saved rollouts:', [Path(p).name for p in roll_paths])
assert roll_paths, 'No rollouts found. Wait for train_lob.py to reach save_every_steps.'

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)
for rp in roll_paths[-3:]:
    decoded = np.load(rp)   # (horizon, FEATURE_DIM_FLAT) in *normalised* units
    step = int(Path(rp).stem.split('_')[-1])
    # Inverse normalize tick-scalar features for plotting
    tick_norm = decoded[:, LEVEL_FLAT:LEVEL_FLAT + F_TICK]
    tick_raw = tick_norm * stats.per_tick_std + stats.per_tick_mean
    axes[0].plot(tick_raw[:, 0], label=f'step {step}')     # mid
    axes[1].plot(tick_raw[:, 1], label=f'step {step}')     # spread
    axes[2].plot(tick_raw[:, 3], label=f'step {step}')     # imbalance
for ax, name in zip(axes, ['mid', 'spread', 'imbalance']):
    ax.set_title(f'imagined {name}'); ax.set_xlabel('imagined step'); ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Sanity assertions
last_roll = np.load(roll_paths[-1])
tick_norm = last_roll[:, LEVEL_FLAT:LEVEL_FLAT + F_TICK]
tick_raw = tick_norm * stats.per_tick_std + stats.per_tick_mean

train_mid_std = seq.per_tick[:, 0].std()
pred_mid_std = tick_raw[:, 0].std()
ratio = pred_mid_std / max(train_mid_std, 1e-6)
print(f'train mid std:   {train_mid_std:.5f}')
print(f'predicted std:   {pred_mid_std:.5f}  (ratio {ratio:.2f}, expect 0.3--3.0)')
print(f'predicted spread > 0 everywhere: {(tick_raw[:, 1] > 0).all()}')
print(f'predicted imbalance in [0, 1]: {(tick_raw[:, 3] >= -0.05).all() and (tick_raw[:, 3] <= 1.05).all()}')